# Multi-Agent Sales Workflow with watsonx.governance

This notebook demonstrates the multi-agent sales workflow with integrated watsonx.governance evaluation.

## Governance Metrics

- **Email Generation Quality:** Faithfulness and professionalism of generated emails
- **Overall Workflow Quality:** Completeness and accuracy of recommendations

## Setup

In [1]:
import os
from dotenv import load_dotenv
import pandas as pd
import uuid

load_dotenv()

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

## Initialize Multi-Agent System

In [2]:
from supervisory_agent import SupervisoryAgent

# MODEL OPTIONS (if you encounter connection errors, try alternatives):
# - "meta-llama/llama-3-3-70b-instruct"  (Primary - Best performance)
# - "meta-llama/llama-3-1-70b-instruct"  (Alternative 1)
# - "meta-llama/llama-3-1-8b-instruct"   (Alternative 2 - Faster, lower cost)
# - "ibm/granite-13b-chat-v2"            (Alternative 3 - IBM model)

supervisor = SupervisoryAgent(
    model_id="meta-llama/llama-3-3-70b-instruct",  # Change this if needed
    url=os.getenv("WATSONX_URL", "https://us-south.ml.cloud.ibm.com"),
    apikey=os.getenv("WATSONX_APIKEY"),
    project_id=os.getenv("WATSONX_PROJECT_ID"),
    contract_vector_store_path="./contract_vector_store",
    crm_file_path="docs/Confluent Sales Cloud Infor.xlsx"
)

print("[SUCCESS] Multi-agent system initialized")

[SUCCESS] Multi-agent system initialized


## Initialize watsonx.governance Evaluator

In [3]:
from ibm_watsonx_gov.evaluators.metrics_evaluator import MetricsEvaluator
from ibm_watsonx_gov.metrics import FaithfulnessMetric
from ibm_watsonx_gov.config import GenAIConfiguration
from ibm_watsonx_gov.entities.foundation_model import WxAIFoundationModel
from ibm_watsonx_gov.entities.llm_judge import LLMJudge

PROJECT_ID = os.getenv("WATSONX_PROJECT_ID")
REGION = "us-south"  # Explicitly set region

llm_judge = LLMJudge(
    model=WxAIFoundationModel(
        model_id="meta-llama/llama-3-3-70b-instruct",
        project_id=PROJECT_ID,
        region=REGION
    )
)

evaluator = MetricsEvaluator(
    project_id=PROJECT_ID,
    region=REGION
)

print(f"[SUCCESS] Governance evaluator initialized (region: {REGION})")

/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/ibm_watsonx_gov/tools/utils/package_utils.py:14: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/ibm_watsonx_gov/tools/utils/environment.py:55: UserWarning: Since WATSONX_REGION is not provided in the environment variable, the Dallas region will be used as the default.
  warnings.warn(
[2026-04-20 13:57:12,594]-[ibm_watsonx_gov.evaluators.agentic_evaluator]-[ WARNING ]-[Line 125] ~~> No module named 'ibm_agent_analytics'


[SUCCESS] Governance evaluator initialized (region: us-south)


## Create Test Queries

In [4]:
# Using only query 0 for focused governance evaluation
test_queries = [
    "I'm a new seller at IBM. I recently got Confluent as a new customer and I want to understand what contracts are coming up for renewal. Are there any contracts that have already expired. Based on the CRM, Contracts and webscraped information can you then put a plan of next steps"
]

query_df = pd.DataFrame({"input_text": test_queries})
query_df["message_id"] = [str(uuid.uuid4()) for _ in range(len(query_df))]
query_df["partner_name"] = "Confluent"
print(f"Testing with {len(query_df)} query for governance evaluation")
query_df

Testing with 1 query for governance evaluation


,input_text,message_id,partner_name
0,"I'm a new seller at IBM. I recently got Confluent as a new customer and I want to understand what contracts are coming up for renewal. Are there any contracts that have already expired. Based on the CRM, Contracts and webscraped information can you then put a plan of next steps",2a746571-ad65-4acd-82dc-879d32fbcc49,Confluent


## Run Workflow with Governance Evaluation

In [5]:
batch_results = []
agent_results = []

for idx, row in query_df.iterrows():
    print("="*80)
    print(f"Processing Query {idx+1}/{len(query_df)}")
    print("="*80)
    print(f"Query: {row['input_text'][:100]}...")
    print()
    
    try:
        result = supervisor.run(
            seller_query=row["input_text"],
            contract_file_path=None,
            partner_name=row["partner_name"]
        )
        
        action_rec = result.get("action_recommendation", {})
        draft_email = action_rec.get("draft_email", "")
        risk_level = action_rec.get("risk_assessment", {}).get("risk_level", "Unknown")
        
        # Store agent result
        agent_results.append({
            "message_id": row["message_id"],
            "input_text": row["input_text"],
            "generated_text": draft_email,
            "risk_level": risk_level,
            "context": str(result.get("contract_summary", {}))[:500] + " | " + str(result.get("partner_profile", {}))[:500]
        })
        
        print(f"[SUCCESS] Workflow completed")
        print(f"Risk Level: {risk_level}")
        print(f"Email Length: {len(draft_email)} characters")
        
    except Exception as e:
        print(f"[ERROR] {str(e)}")
        agent_results.append({
            "message_id": row["message_id"],
            "input_text": row["input_text"],
            "generated_text": f"Error: {str(e)}",
            "risk_level": "Error",
            "context": ""
        })
    
    print("\n")

agent_df = pd.DataFrame(agent_results)

print("="*80)
print("BATCH PROCESSING COMPLETE")
print("="*80)

Processing Query 1/1
Query: I'm a new seller at IBM. I recently got Confluent as a new customer and I want to understand what co...


SUPERVISORY AGENT - Workflow Initialization
Seller Query: I'm a new seller at IBM. I recently got Confluent as a new customer and I want to understand what contracts are coming up for renewal. Are there any contracts that have already expired. Based on the CRM, Contracts and webscraped information can you then put a plan of next steps

Workflow Type: renewal_expiration_awareness
Required Agents: contract, action
Partner Name: Confluent

EXECUTING CONTRACT AGENT
Preloading contract portfolio for partner: Confluent
Contract scope: all files in docs/ beginning with Confluent_IBM
✓ Using cached data for: Confluent_IBM-1.30.2024.docx
✓ Using cached metadata for: Confluent_IBM-1.30.2024.docx
✓ Using cached structured data for: Confluent_IBM-1.30.2024.docx
✓ Using cached data for: Confluent_IBM-1.30.2025.docx
✓ Using cached metadata for: Confluent_IBM-1.30.2025

Failure during generate. (POST https://us-south.ml.cloud.ibm.com/ml/v1/text/generation?version=2025-11-12)
Status code: 502, body: <html>
<head><title>502 Bad Gateway</title></head>
<body>
<center><h1>502 Bad Gateway</h1></center>
<hr><center>cloudflare</center>
</body>
</html>




Research Agent error: Failure during generate. (POST https://us-south.ml.cloud.ibm.com/ml/v1/text/generation?version=2025-11-12)
Status code: 502, body: <html>
<head><title>502 Bad Gateway</title></head>
<body>
<center><h1>502 Bad Gateway</h1></center>
<hr><center>cloudflare</center>
</body>
</html>


EXECUTING MATCHING AGENT

LLM Matching for Confluent_IBM-5.30.2023.docx:
  Matched: 0 opportunities
  Confidence: 
  Reasoning: ...


Failure during generate. (POST https://us-south.ml.cloud.ibm.com/ml/v1/text/generation?version=2025-11-12)
Status code: 500, body: {"errors":[{"code":"downstream_request_failed","message":"Downstream vllm request with Model 'meta-llama/llama-3-3-70b-instruct failed: Post \"http://\u003chost\u003e:\u003cport\u003e/v1/completions\": dial tcp [::1]:3000: connect: connection refused","more_info":"https://cloud.ibm.com/apidocs/watsonx-ai#text-generation"}],"trace":"6edf162cba28413c12321ad67f6a1397","status_code":500}


LLM matching error: Failure during generate. (POST https://us-south.ml.cloud.ibm.com/ml/v1/text/generation?version=2025-11-12)
Status code: 500, body: {"errors":[{"code":"downstream_request_failed","message":"Downstream vllm request with Model 'meta-llama/llama-3-3-70b-instruct failed: Post \"http://\u003chost\u003e:\u003cport\u003e/v1/completions\": dial tcp [::1]:3000: connect: connection refused","more_info":"https://cloud.ibm.com/apidocs/watsonx-ai#text-generation"}],"trace":"6edf162cba28413c12321ad67f6a1397","status_code":500}

LLM Matching for Confluent_IBM-1.30.2024.docx:
  Matched: 0 opportunities
  Confidence: low
  Reasoning: There are no CRM opportunities provided to match against the given contract, making it impossible to...

LLM Matching for Confluent_IBM-1.30.2025.docx:
  Matched: 0 opportunities
  Confidence: high
  Reasoning: The contract is expired, and I applied the matching rules for expired contracts. Opportunities 1, 4,...
  Matching complete: 0 matched, 4 unmatch

## Extract Next Steps and High Urgency Actions for Evaluation

In [6]:
print("="*80)
print("EXTRACTING NEXT STEPS FOR GOVERNANCE EVALUATION")
print("="*80)

# Extract next steps and high urgency actions from workflow results
next_steps_results = []

for idx, row in query_df.iterrows():
    try:
        result = supervisor.run(
            seller_query=row["input_text"],
            contract_file_path=None,
            partner_name=row["partner_name"]
        )
        
        action_rec = result.get("action_recommendation", {})
        ranked_next_steps = action_rec.get("ranked_next_steps", [])
        detailed_actions = action_rec.get("detailed_actions", [])
        
        # Get high urgency next step (top priority)
        high_urgency_step = ranked_next_steps[0] if ranked_next_steps else "No action identified"
        
        # Get all next steps as a single text
        all_next_steps = " | ".join(ranked_next_steps[:5]) if ranked_next_steps else "No next steps identified"
        
        # Extract context from contract and CRM data
        context_parts = []
        if detailed_actions:
            top_action = detailed_actions[0]
            context_parts.append(f"Contract: {top_action.get('contract', 'Unknown')}")
            context_parts.append(f"Urgency: {top_action.get('urgency_level', 'Unknown')}")
            context_parts.append(f"Priority: {top_action.get('priority', 'Unknown')}")
            if top_action.get('recipient_info'):
                recipient = top_action['recipient_info']
                context_parts.append(f"Recipient: {recipient.get('name', recipient.get('role', 'Unknown'))}")
        
        context_text = " | ".join(context_parts) if context_parts else str(result.get("contract_summary", {}))[:500]
        
        next_steps_results.append({
            "message_id": row["message_id"],
            "input_text": row["input_text"],
            "high_urgency_step": high_urgency_step,
            "all_next_steps": all_next_steps,
            "context": context_text
        })
        
        print(f"[SUCCESS] Extracted next steps for query {idx+1}")
        
    except Exception as e:
        print(f"[ERROR] Failed to extract next steps for query {idx+1}: {str(e)}")
        next_steps_results.append({
            "message_id": row["message_id"],
            "input_text": row["input_text"],
            "high_urgency_step": f"Error: {str(e)}",
            "all_next_steps": f"Error: {str(e)}",
            "context": ""
        })

next_steps_df = pd.DataFrame(next_steps_results)
print(f"\n[SUCCESS] Extracted next steps for {len(next_steps_df)} queries")

EXTRACTING NEXT STEPS FOR GOVERNANCE EVALUATION

SUPERVISORY AGENT - Workflow Initialization
Seller Query: I'm a new seller at IBM. I recently got Confluent as a new customer and I want to understand what contracts are coming up for renewal. Are there any contracts that have already expired. Based on the CRM, Contracts and webscraped information can you then put a plan of next steps


Failure during generate. (POST https://us-south.ml.cloud.ibm.com/ml/v1/text/generation?version=2025-11-12)
Status code: 500, body: {"errors":[{"code":"downstream_request_failed","message":"Downstream vllm request with Model 'meta-llama/llama-3-3-70b-instruct failed: Post \"http://\u003chost\u003e:\u003cport\u003e/v1/completions\": dial tcp [::1]:3000: connect: connection refused","more_info":"https://cloud.ibm.com/apidocs/watsonx-ai#text-generation"}],"trace":"f6128a38dcb3431d96230f273f6ddf63","status_code":500}



Workflow Type: renewal_expiration_awareness
Required Agents: contract, action
Partner Name: Confluent

EXECUTING CONTRACT AGENT
Preloading contract portfolio for partner: Confluent
Contract scope: all files in docs/ beginning with Confluent_IBM
✓ Using cached data for: Confluent_IBM-1.30.2024.docx
✓ Using cached metadata for: Confluent_IBM-1.30.2024.docx
✓ Using cached structured data for: Confluent_IBM-1.30.2024.docx
✓ Using cached data for: Confluent_IBM-1.30.2025.docx
✓ Using cached metadata for: Confluent_IBM-1.30.2025.docx
✓ Using cached structured data for: Confluent_IBM-1.30.2025.docx
✓ Using cached data for: Confluent_IBM-5.30.2023.docx
✓ Using cached metadata for: Confluent_IBM-5.30.2023.docx
✓ Using cached structured data for: Confluent_IBM-5.30.2023.docx
✓ Using cached data for: Confluent_IBM-7.31.2024.docx
✓ Using cached metadata for: Confluent_IBM-7.31.2024.docx
✓ Using cached structured data for: Confluent_IBM-7.31.2024.docx

Contract Agent completed successfully
Contrac

Failure during generate. (POST https://us-south.ml.cloud.ibm.com/ml/v1/text/generation?version=2025-11-12)
Status code: 500, body: {"errors":[{"code":"downstream_request_failed","message":"Downstream vllm request with Model 'meta-llama/llama-3-3-70b-instruct failed: Post \"http://\u003chost\u003e:\u003cport\u003e/v1/completions\": dial tcp [::1]:3000: connect: connection refused","more_info":"https://cloud.ibm.com/apidocs/watsonx-ai#text-generation"}],"trace":"62ee939f4f9195d11d087e99768c2436","status_code":500}


LLM matching error: Failure during generate. (POST https://us-south.ml.cloud.ibm.com/ml/v1/text/generation?version=2025-11-12)
Status code: 500, body: {"errors":[{"code":"downstream_request_failed","message":"Downstream vllm request with Model 'meta-llama/llama-3-3-70b-instruct failed: Post \"http://\u003chost\u003e:\u003cport\u003e/v1/completions\": dial tcp [::1]:3000: connect: connection refused","more_info":"https://cloud.ibm.com/apidocs/watsonx-ai#text-generation"}],"trace":"62ee939f4f9195d11d087e99768c2436","status_code":500}


Failure during generate. (POST https://us-south.ml.cloud.ibm.com/ml/v1/text/generation?version=2025-11-12)
Status code: 500, body: {"errors":[{"code":"downstream_request_failed","message":"Downstream vllm request with Model 'meta-llama/llama-3-3-70b-instruct failed: Post \"http://\u003chost\u003e:\u003cport\u003e/v1/completions\": dial tcp [::1]:3000: connect: connection refused","more_info":"https://cloud.ibm.com/apidocs/watsonx-ai#text-generation"}],"trace":"9781df4ed7fc0b85108d2c54e8a37305","status_code":500}


LLM matching error: Failure during generate. (POST https://us-south.ml.cloud.ibm.com/ml/v1/text/generation?version=2025-11-12)
Status code: 500, body: {"errors":[{"code":"downstream_request_failed","message":"Downstream vllm request with Model 'meta-llama/llama-3-3-70b-instruct failed: Post \"http://\u003chost\u003e:\u003cport\u003e/v1/completions\": dial tcp [::1]:3000: connect: connection refused","more_info":"https://cloud.ibm.com/apidocs/watsonx-ai#text-generation"}],"trace":"9781df4ed7fc0b85108d2c54e8a37305","status_code":500}

LLM Matching for Confluent_IBM-1.30.2024.docx:
  Matched: 2 opportunities
  Confidence: high
  Reasoning: The contract has an amount of $250,003.20, which matches the amount of $250,000 in opportunities 3 a...


KeyboardInterrupt: 

## Evaluate Next Steps Quality with Governance

In [ ]:
print("\n" + "="*80)
print("EVALUATING NEXT STEPS QUALITY WITH WATSONX.GOVERNANCE")
print("="*80)

# Filter out error results
valid_next_steps = next_steps_df[~next_steps_df["high_urgency_step"].str.contains("Error", na=False)].copy()

if not valid_next_steps.empty:
    # Prepare evaluation data for high urgency steps
    eval_urgency_data = valid_next_steps[["input_text", "context"]].copy()
    eval_urgency_data["generated_text"] = valid_next_steps["high_urgency_step"]
    
    # Create faithfulness metric for next steps
    config_urgency = GenAIConfiguration(
        input_fields=["input_text"],
        context_fields=["context"],
        output_fields=["generated_text"]
    )
    
    faithfulness_urgency = FaithfulnessMetric(
        llm_judge=llm_judge,
        configuration=config_urgency
    )
    
    try:
        print(f"\nEvaluating {len(eval_urgency_data)} high urgency next steps...")
        eval_urgency_result = evaluator.evaluate(
            data=eval_urgency_data,
            metrics=[faithfulness_urgency]
        )
        
        if eval_urgency_result and hasattr(eval_urgency_result, 'to_df'):
            urgency_metrics_df = eval_urgency_result.to_df()
            
            if not urgency_metrics_df.empty:
                print(f"\n[SUCCESS] Next steps evaluation complete")
                
                # Add message_ids to metrics
                urgency_metrics_df["message_id"] = valid_next_steps["message_id"].values
                
                # Merge with next steps results
                next_steps_with_metrics = next_steps_df.merge(urgency_metrics_df, on="message_id", how="left")
                
                print("\n" + "="*80)
                print("NEXT STEPS EVALUATION SUMMARY")
                print("="*80)
                
                # Find score column
                score_col = None
                for col in ['value', 'score', 'metric_value']:
                    if col in urgency_metrics_df.columns:
                        score_col = col
                        break
                
                if score_col:
                    avg_score = urgency_metrics_df[score_col].mean()
                    min_score = urgency_metrics_df[score_col].min()
                    max_score = urgency_metrics_df[score_col].max()
                    
                    print(f"Average Next Steps Faithfulness Score: {avg_score:.2f}")
                    print(f"Min Score: {min_score:.2f}")
                    print(f"Max Score: {max_score:.2f}")
                    
                    if avg_score >= 0.8:
                        print("\nNext Steps Quality Assessment: EXCELLENT")
                    elif avg_score >= 0.6:
                        print("\nNext Steps Quality Assessment: GOOD")
                    elif avg_score >= 0.4:
                        print("\nNext Steps Quality Assessment: FAIR")
                    else:
                        print("\nNext Steps Quality Assessment: NEEDS IMPROVEMENT")
                else:
                    print("[WARNING] Could not find score column in next steps evaluation")
                    next_steps_with_metrics = next_steps_df
            else:
                print("[WARNING] No metrics returned from next steps evaluation")
                next_steps_with_metrics = next_steps_df
        else:
            print("[WARNING] Next steps evaluation returned no result")
            next_steps_with_metrics = next_steps_df
            
    except Exception as e:
        print(f"[ERROR] During next steps evaluation: {str(e)}")
        import traceback
        traceback.print_exc()
        next_steps_with_metrics = next_steps_df
else:
    print("[WARNING] No valid next steps to evaluate")
    next_steps_with_metrics = next_steps_df


EVALUATING NEXT STEPS QUALITY WITH WATSONX.GOVERNANCE

Evaluating 1 high urgency next steps...
[Warning] No region provided : Using default region as us-south

[SUCCESS] Next steps evaluation complete

NEXT STEPS EVALUATION SUMMARY
[WARNING] Could not find score column in next steps evaluation


## Evaluate Email Quality with Governance

In [ ]:
print("="*80)
print("EVALUATING EMAIL QUALITY WITH WATSONX.GOVERNANCE")
print("="*80)

# Filter out error results
valid_results = agent_df[agent_df["risk_level"] != "Error"].copy()

if not valid_results.empty:
    # Prepare evaluation data
    eval_data = valid_results[["input_text", "context", "generated_text"]].copy()
    
    # Create faithfulness metric with configuration
    config = GenAIConfiguration(
        input_fields=["input_text"],
        context_fields=["context"],
        output_fields=["generated_text"]
    )
    
    faithfulness_metric = FaithfulnessMetric(
        llm_judge=llm_judge,
        configuration=config
    )
    
    try:
        print(f"\nEvaluating {len(eval_data)} emails...")
        eval_result = evaluator.evaluate(
            data=eval_data,
            metrics=[faithfulness_metric]
        )
        
        if eval_result and hasattr(eval_result, 'to_df'):
            metrics_df = eval_result.to_df()
            
            if not metrics_df.empty:
                print(f"\n[SUCCESS] Evaluation complete")
                print(f"\nMetrics DataFrame:")
                print(metrics_df)
                
                # Add message_ids to metrics
                metrics_df["message_id"] = valid_results["message_id"].values
                
                # Merge with agent results
                final_df = agent_df.merge(metrics_df, on="message_id", how="left")
                
                print("\n" + "="*80)
                print("EVALUATION SUMMARY")
                print("="*80)
                
                # Find score column
                score_col = None
                for col in ['value', 'score', 'metric_value']:
                    if col in metrics_df.columns:
                        score_col = col
                        break
                
                if score_col:
                    avg_score = metrics_df[score_col].mean()
                    min_score = metrics_df[score_col].min()
                    max_score = metrics_df[score_col].max()
                    
                    print(f"Average Faithfulness Score: {avg_score:.2f}")
                    print(f"Min Score: {min_score:.2f}")
                    print(f"Max Score: {max_score:.2f}")
                    
                    if avg_score >= 0.8:
                        print("\nOverall Assessment: EXCELLENT")
                    elif avg_score >= 0.6:
                        print("\nOverall Assessment: GOOD")
                    elif avg_score >= 0.4:
                        print("\nOverall Assessment: FAIR")
                    else:
                        print("\nOverall Assessment: NEEDS IMPROVEMENT")
                else:
                    print("[WARNING] Could not find score column in results")
                    final_df = agent_df
            else:
                print("[WARNING] No metrics returned from evaluation")
                final_df = agent_df
        else:
            print("[WARNING] Evaluation returned no result")
            final_df = agent_df
            
    except Exception as e:
        print(f"[ERROR] During evaluation: {str(e)}")
        import traceback
        traceback.print_exc()
        final_df = agent_df
else:
    print("[WARNING] No valid results to evaluate")
    final_df = agent_df

EVALUATING EMAIL QUALITY WITH WATSONX.GOVERNANCE

Evaluating 1 emails...

[SUCCESS] Evaluation complete

Metrics DataFrame:
   faithfulness.llm_as_judge
0                        0.0

EVALUATION SUMMARY
[WARNING] Could not find score column in results


## View Results

In [ ]:
print("="*80)
print("FINAL RESULTS")
print("="*80)
display(final_df)

FINAL RESULTS


,message_id,input_text,generated_text,risk_level,context
0,ca60613b-c762-48eb-94c0-4a1170097265,"I'm a new seller at IBM. I recently got Confluent as a new customer and I want to understand what contracts are coming up for renewal. Are there any contracts that have already expired. Based on the CRM, Contracts and webscraped information can you then put a plan of next steps","Subject: Following up on watsonx renewal\n\nHi Rohan,\n\nIt was great connecting and I wanted to check in on the watsonx renewal we previously discussed. Have you finalized the sizing for this agreement? I'm looking forward to moving things forward.\n\nOur contract has expired, so I'd like to help get the renewal in place quickly to avoid any delays. Would it be helpful to schedule a call to discuss any questions you may have? I'll be coordinating with our team to ensure a smooth process. \nRegards,\n[Your Name]",High,"{'partner_name': 'Confluent', 'contract_paths': ['docs/Confluent_IBM-1.30.2024.docx', 'docs/Confluent_IBM-1.30.2025.docx', 'docs/Confluent_IBM-5.30.2023.docx', 'docs/Confluent_IBM-7.31.2024.docx'], 'contract_results': [{'file_path': 'docs/Confluent_IBM-1.30.2024.docx', 'file_name': 'Confluent_IBM-1.30.2024.docx', 'structured_summary': {'amount': '$250,003.20', 'amount_numeric': 250003.2, 'products': ['watsonx'], 'start_date': 'Jan 31, 2024', 'term_length': '1 year(s)', 'coverage_period_start': ' | {'partner_name': 'Confluent', 'maturity_level': 'Engaged Prospect', 'sales_velocity': 'High', 'deal_blockers': [{'opportunity': 'Confluent watsonx ESA Expansion Upside', 'reason': 'Did not want to go with the expansion'}, {'opportunity': 'Cognos Usage', 'reason': 'CPO Not ready to move forward yet'}, {'opportunity': 'Confluent watsonx ESA', 'reason': 'Needed to delay renewal due to change in org structure'}], 'external_signals': {'query': 'Confluent company background news technology partnership"


## View Sample Email

In [ ]:
if not final_df.empty:
    sample_idx = 0
    sample = final_df.iloc[sample_idx]
    
    print("="*80)
    print("SAMPLE EMAIL")
    print("="*80)
    print(f"\nQuery: {sample['input_text']}")
    print(f"\nRisk Level: {sample['risk_level']}")
    
    # Show score if available
    for col in ['value', 'score', 'metric_value']:
        if col in sample and pd.notna(sample[col]):
            print(f"Faithfulness Score: {sample[col]:.2f}")
            break
    
    print("\n" + "-"*80)
    print("DRAFT EMAIL:")
    print("-"*80)
    print(sample["generated_text"])
else:
    print("No results to display")

SAMPLE EMAIL

Query: I'm a new seller at IBM. I recently got Confluent as a new customer and I want to understand what contracts are coming up for renewal. Are there any contracts that have already expired. Based on the CRM, Contracts and webscraped information can you then put a plan of next steps

Risk Level: High

--------------------------------------------------------------------------------
DRAFT EMAIL:
--------------------------------------------------------------------------------
Subject: Following up on watsonx renewal

Hi Rohan,

It was great connecting and I wanted to check in on the watsonx renewal we previously discussed. Have you finalized the sizing for this agreement? I'm looking forward to moving things forward.

Our contract has expired, so I'd like to help get the renewal in place quickly to avoid any delays. Would it be helpful to schedule a call to discuss any questions you may have? I'll be coordinating with our team to ensure a smooth process. 
Regards,
[Your Na

## Summary

This notebook demonstrates:
1. **Multi-agent workflow execution** - Processes seller queries through contract, research, matching, and action agents
2. **Email generation** - Creates professional outreach emails based on contract and CRM data
3. **Governance evaluation** - Uses watsonx.governance to assess email quality and faithfulness
4. **Batch processing** - Evaluates multiple queries and aggregates results

### Next Steps:
- Adjust evaluation metrics based on your quality requirements
- Integrate into production workflow for continuous quality monitoring
- Add additional metrics (e.g., ContextRelevanceMetric, HallucinationMetric)